# M03.04 — When the answer is a picture

The six policy PDFs, and four figures of statistics (`data/insurance_figures/`). The answers to our questions are only in the figures: the PDFs never state them. A text index cannot answer; we index the figures through captions written by a model that reads images, and let that model read the value off the retrieved figure.

| stage | tool |
|---|---|
| indexing | `gpt-6-luna` captions each figure, `text-embedding-3-small` embeds the captions |
| retrieval | the same `MiniVectorStore` as for text |
| generation | `gpt-6-luna`, reading the retrieved image |

The figures are synthetic (generated by `data/generate_data.ipynb`), so the ground truth is exact.

## Setup

In [ ]:
import json

from IPython.display import Image, display
from agentic_workshop.rag import (
    answer,
    build_store,
    embed,
    load_documents,
    retrieve,
    show,
    split_structured,
)

from agentic_workshop.client import get_client
from agentic_workshop.config import CHAT_MODEL, DATA_DIR
from agentic_workshop.invoices import image_to_data_url
from agentic_workshop.vector_store import MiniVectorStore

client = get_client()
MODEL = CHAT_MODEL
FIGURES_DIR = DATA_DIR / "insurance_figures"

## 1. The figures

One question per figure, with its answer, computed from the data behind the chart.

In [ ]:
figure_questions = json.loads((FIGURES_DIR / "ground_truth.json").read_text(encoding="utf-8"))
figure_paths = sorted(FIGURES_DIR.glob("*.png"))

for q in figure_questions:
    print(f"{q['figure']:28s} {q['question']}\n{'':28s} -> {q['answer']}")

display(Image(filename=FIGURES_DIR / "fig_premiums.png", width=450))

## 2. The text pipeline cannot answer this

The structured-split store of M03.02, over the six policy PDFs.

In [ ]:
question = figure_questions[0]["question"]

text_store = build_store(load_documents(), split_structured)
show(retrieve(text_store, question, k=3), preview=0)
print()
print(answer(text_store, question, k=3, instruction="If the extracts do not contain the answer, say so plainly."))

Retrieval returns the closest articles, about premiums and insurance models, and the model correctly says the answer isn't there. The figures were never indexed. With scanned PDFs it would be worse: stray axis labels would be retrieved as if they meant something.

## 3. Index the figures through captions

An embedding model for text can't embed an image. But a model that reads images can **describe** it, and the description is text: we embed it like any chunk. The caption should name everything a question could ask about: the title, the chart type, the series and labels, and the values.

In [ ]:
CAPTION_PROMPT = (
    "Describe this chart for a search index. Give its title, the chart type, what the axes or segments "
    "show, every series or category label, and the values you can read. Plain text, no Markdown."
)


def caption(path) -> str:
    response = client.responses.create(model=MODEL, input=[{"role": "user", "content": [
        {"type": "input_text", "text": CAPTION_PROMPT},
        {"type": "input_image", "image_url": image_to_data_url(path)},
    ]}])
    return response.output_text


captions = {path.name: caption(path) for path in figure_paths}
print(captions["fig_premiums.png"])

In [ ]:
figure_store = MiniVectorStore()  # the same class as for text: agentic_workshop/vector_store.py
figure_store.add(
    embeddings=embed(list(captions.values())),
    texts=list(captions.values()),
    metadata=[{"source": name, "path": FIGURES_DIR / name} for name in captions],
)

hits = figure_store.query(embed([question])[0], k=4)
show(hits, preview=0)

The question is matched against the captions, so we can query with the question itself. The ranking is only as good as the captions: what the captioning model didn't write down can't be found.

## 4. A multimodal model reads the figure

A caption may round or miss values, so the answer comes from the image itself: we send the retrieved figure to the model.

In [ ]:
def answer_from_figure(question: str, path) -> str:
    response = client.responses.create(model=MODEL, input=[{"role": "user", "content": [
        {"type": "input_text", "text": "Answer the question using only this figure. "
                                       f"Quote the values you read off it.\n\nQUESTION: {question}"},
        {"type": "input_image", "image_url": image_to_data_url(path)},
    ]}])
    return response.output_text


best = hits[0]["metadata"]
print(f"reading: {best['source']}\n")
print(answer_from_figure(question, best["path"]))
print(f"\nground truth: {figure_questions[0]['answer']}")

Now the whole pipeline on every question: retrieve the top figure from its caption, read it, and check the answer against the ground truth.

In [ ]:
def visual_rag(question: str) -> tuple[str, str]:
    """(figure retrieved, answer read off it)"""
    figure = figure_store.query(embed([question])[0], k=1)[0]["metadata"]
    return figure["source"], answer_from_figure(question, figure["path"])


for q in figure_questions:
    retrieved, answer_text = visual_rag(q["question"])
    # A rough check: "5" also matches "2025". A stricter one matches whole words only:
    # all(re.search(rf"\b{re.escape(s)}\b", answer_text) for s in q["expected"])
    correct = all(s in answer_text for s in q["expected"])
    print(f"Q: {q['question']}")
    print(f"   retrieved {retrieved} ({'right' if retrieved == q['figure'] else 'WRONG'} figure)")
    print(f"   {answer_text}")
    print(f"   expected {q['answer']}: {'contains it' if correct else 'CHECK'}\n")

## Takeaways

- A text index cannot answer from a figure, and the failure is silent: retrieval looks fine.
- A model that reads images turns a figure into text (a caption) that an ordinary embedding model can index.
- Retrieval and reading are separate jobs: the caption finds the figure, the model reads the value off the image itself.
- The captions decide what can be found. Index them like any chunk, and check them like any chunk.

## Extensions

**Easy**

1. **Caption quality.** Write a caption prompt that only gives the title and chart type. Which questions stop finding their figure? Then ask questions the captions miss (e.g. about a canton not labelled).
2. **Mixed store.** Put the captions in the same store as the text pieces of section 2. Does a question about premiums now retrieve the figure, the articles, or both?

**Medium**

3. **CLIP: a true image embedder.** CLIP embeds images and text into one vector space, with no caption. Install it with `uv sync --group visual` (about 600 MB of model weights on first run), then run the cell below. CLIP matches images against *captions*, not questions: query it with a description of the figure, and keep that description short (its text encoder reads at most 77 tokens). Compare its ranking with the caption store's.
4. **Real documents.** Take a real report with charts (e.g. from the Federal Office of Public Health), extract its images with `pypdf` (`page.images`), caption and index them. Compare with indexing the extracted text.

**Hard**

5. **Late interaction.** Read up on ColPali. Approximate it with a 3×3 grid of CLIP tiles per figure, then try `colpali-engine`. Compare accuracy, index size and latency.

In [ ]:
# Extension 3: CLIP (needs `uv sync --group visual`)
import numpy as np
from PIL import Image as PILImage
from sentence_transformers import SentenceTransformer


def load_rgb(path) -> PILImage.Image:
    with PILImage.open(path) as image:
        return image.convert("RGB")


clip = SentenceTransformer("clip-ViT-B-32")
clip_store = MiniVectorStore()
clip_store.add(
    embeddings=clip.encode([load_rgb(p) for p in figure_paths]),
    texts=[p.name for p in figure_paths],
    metadata=[{"source": p.name, "path": p} for p in figure_paths],
)


def to_visual_query(question: str) -> str:
    """Rewrite a question as a short description of the figure that would answer it."""
    response = client.responses.create(model=MODEL, input=(
        "Rewrite the question as a short visual description of the chart that would answer it, as you "
        "would caption an image, in at most 20 words. Describe the chart type and what is plotted. "
        f"Do not answer the question. Reply with the description only.\n\nQUESTION: {question}"))
    return response.output_text.strip()


for q in figure_questions:
    visual_query = to_visual_query(q["question"])
    hit = clip_store.query(np.asarray(clip.encode(visual_query)), k=1)[0]
    print(f"{q['figure']:28s} question -> {clip_store.query(np.asarray(clip.encode(q['question'])), k=1)[0]['text']:28s} "
          f"description -> {hit['text']}  ({visual_query!r})")